 # 03b_RLVR_Manual_Analysis_v24.py



 ### 🔬 Institutional Regime Attribution, Beta Forensics & Factor Allocation Deep-Dive



 - **Leaderboard & Auto-Discovery**: Audits all continuous walk-forward blotters across generations (`gen17`–`gen21`+).

 - **Universal Target Selection**: Interactive and programmatic target selection defaulting to the latest generation ex-post blend (`blotter_continuous_ex_post_blend_gen21.parquet`).

 - **Regime-Partitioned Audit**: Discrete breakdown across the 2022 Rate Shock, 2023 Tech Rally, and 2024–2026 Dispersion regimes.

 - **Forensic Beta Deconstruction**: Mathematically isolates systematic market leverage from idiosyncratic stock-picking alpha.

 - **Action Space & Factor Radar**: Decodes factor weights, tri-asset allocations, and extreme tail risk profiles.

 - **Multi-Seed Arithmetic & Compounding Auditor**: Bit-level proof of linear blend return aggregation and non-leaking geometric equity compounding.

 - **Agent Training Diagnostics (`217_diag`)**: 6-panel PPO convergence telemetry (Losses, Critic Value Loss, Entropy, Approx KL & Clip Fraction, Explained Variance, and Step Reward).

In [1]:
# CELL 1: Environment Setup, Paths, Hardware & Dynamic Config Ingestion
import gc
import os
import pickle
import re
import sys
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple, Union

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import torch

# Environment & Path Resolution
RUNNING_IN_COLAB = "google.colab" in sys.modules
if RUNNING_IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    rl_root = Path(
        "/content/drive/Othercomputers/My Computer/Files_win10/python/py311/stocks/notebooks_RLVR_v3"
    )
else:
    rl_root_default = Path(
        r"C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v3"
    )
    rl_root = rl_root_default if rl_root_default.exists() else Path.cwd()

if str(rl_root) not in sys.path:
    sys.path.append(str(rl_root))
os.chdir(rl_root)

from core.paths import LOCAL_DATA_DIR, OUTPUT_DIR
from core.settings import CacheConfig, TradingConfig
from data_pipeline.loader import load_processed_data
from data_pipeline.utils import get_master_trading_calendar
from rl_discovery.adapter import RLVRGymEnv
from rl_discovery.agent import AbsoluteZeroAgent
from rl_discovery.environment import DiscoveryEnv
from rl_discovery.validator import AgentEvaluator
from strategy.registry import get_strategy_registry

checkpoint_dir = OUTPUT_DIR / "model_checkpoints"
base_config = TradingConfig()
benchmark_name = base_config.benchmark
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"📁 Workspace Root : {rl_root}")
print(f"📁 Output Vault    : {OUTPUT_DIR}")
print(f"⚙️ Compute Device  : {device}")
print(
    f"🏛️ Benchmark       : {benchmark_name} | Holding Period: {base_config.holding_period}d"
)

NOTEBOOKS_RLVR_ROOT: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v3
PROJECT_ROOT: C:\Users\ping\Files_win10\python\py311\stocks
GLOBAL_DATA_DIR: C:\Users\ping\Files_win10\python\py311\stocks\data
GLOBAL_PROCESSED_DIR: C:\Users\ping\Files_win10\python\py311\stocks\data\processed
LOCAL_DATA_DIR: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v3\data
OUTPUT_DIR: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v3\output
TRADES_DIR: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v3\trades

📁 Workspace Root : C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v3
📁 Output Vault    : C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v3\output
⚙️ Compute Device  : cpu
🏛️ Benchmark       : SPY | Holding Period: 5d


In [2]:
# CELL 2: Continuous Walk-Forward Leaderboard & Candidate Discovery
def calculate_blotter_metrics(df_blotter: pd.DataFrame) -> Dict[str, Any]:
    """Computes standardized portfolio performance metrics from an authoritative blotter."""
    b_df = df_blotter.copy()
    date_col = b_df["date"] if "date" in b_df.columns else b_df["decision_date"]
    b_df["date"] = pd.to_datetime(date_col)
    b_df = b_df.sort_values("date").reset_index(drop=True)

    daily_net = b_df["net_daily_simple_ret"].to_numpy(dtype=float)
    daily_bm = b_df["bm_daily_simple_ret"].to_numpy(dtype=float)
    daily_spread = daily_net - daily_bm

    # Strict Geometric Compounding Invariant
    cum_agent = float(np.prod(1.0 + daily_net) - 1.0)
    cum_bm = float(np.prod(1.0 + daily_bm) - 1.0)
    excess_ret = float(cum_agent - cum_bm)

    mean_ret = float(np.mean(daily_net))
    std_ret = float(np.std(daily_net, ddof=1)) if len(daily_net) > 1 else 1e-8
    sharpe = float((mean_ret / std_ret) * np.sqrt(252)) if std_ret > 1e-8 else 0.0

    mean_sp = float(np.mean(daily_spread))
    std_sp = float(np.std(daily_spread, ddof=1)) if len(daily_spread) > 1 else 1e-8
    ir = float((mean_sp / std_sp) * np.sqrt(252)) if std_sp > 1e-8 else 0.0

    downside = daily_spread[daily_spread < 0]
    downside_std = float(np.std(downside, ddof=1)) if len(downside) > 1 else 1e-8
    sortino = (
        float((mean_sp / downside_std) * np.sqrt(252)) if downside_std > 1e-8 else 0.0
    )

    eq = b_df["agent_equity"].to_numpy(dtype=float)
    peaks = np.maximum.accumulate(eq)
    dds = (eq - peaks) / np.maximum(peaks, 1e-8)
    max_dd = float(np.min(dds))

    bm_var = float(np.var(daily_bm, ddof=1)) if len(daily_bm) > 1 else 1e-8
    beta = float(np.cov(daily_net, daily_bm)[0, 1] / bm_var) if bm_var > 1e-8 else 1.0

    return {
        "Total Return": cum_agent,
        "BM Return": cum_bm,
        "Excess Return": excess_ret,
        "Sharpe": sharpe,
        "Sortino": sortino,
        "IR": ir,
        "Max DD": max_dd,
        "Beta": beta,
        "Sessions": len(daily_net),
        "Start Date": pd.to_datetime(b_df["date"].iloc[0]).strftime("%Y-%m-%d"),
        "End Date": pd.to_datetime(b_df["date"].iloc[-1]).strftime("%Y-%m-%d"),
    }


def parse_blotter_sort_key(filename: str) -> Tuple[int, int, int, str]:
    """Deterministic hierarchical sort key:
    1. Generation number (ascending integer, e.g. gen17 -> 17)
    2. Tier rank:
       - 0: ex_post_blend (generation lead champion)
       - 1: constituent seeds (sorted numerically by seed ID)
       - 2: committee or other combination blotters
       - 3: unclassified / fallback
    3. Seed ID (numerical: 42, 101, 777, 1337, 2026)
    4. Canonical filename tie-breaker.
    """
    gen_match = re.search(r"gen(\d+)", filename, re.IGNORECASE)
    gen_num = int(gen_match.group(1)) if gen_match else 9999

    fname_lower = filename.lower()
    if "ex_post_blend" in fname_lower:
        tier_rank = 0
        seed_num = 0
    else:
        seed_match = re.search(r"_s(\d+)(?:_|\.|$)", fname_lower)
        if seed_match:
            tier_rank = 1
            seed_num = int(seed_match.group(1))
        elif "committee" in fname_lower:
            tier_rank = 2
            seed_num = 0
        else:
            tier_rank = 3
            seed_num = 0

    return (gen_num, tier_rank, seed_num, filename)


# Discover all walk-forward continuous blotters in output directory
continuous_blotters = sorted(list(OUTPUT_DIR.glob("blotter_continuous_*.parquet")))
discovered_checkpoints = sorted(
    list({p.resolve(): p for p in OUTPUT_DIR.glob("**/*.pt")}.values()),
    key=lambda p: (parse_blotter_sort_key(p.name)[0], p.name),
)
print(
    f"🔍 Found {len(continuous_blotters)} Continuous Blotters & {len(discovered_checkpoints)} Checkpoints."
)

wf_records = []
for bp in continuous_blotters:
    df_b = pd.read_parquet(bp)
    date_col = "date" if "date" in df_b.columns else "decision_date"
    df_b["date"] = pd.to_datetime(df_b[date_col])
    metrics = calculate_blotter_metrics(df_b)
    metrics["Blotter Name"] = bp.name
    wf_records.append(metrics)

if not wf_records:
    raise FileNotFoundError("⚠️ No continuous blotters detected in output directory.")

df_wf = pd.DataFrame(wf_records)
sort_metadata = [parse_blotter_sort_key(name) for name in df_wf["Blotter Name"]]
df_wf["_gen"] = [m[0] for m in sort_metadata]
df_wf["_tier"] = [m[1] for m in sort_metadata]
df_wf["_seed"] = [m[2] for m in sort_metadata]

# Hierarchical Sort: Generation -> Ex-Post Blend first -> Constituent Seeds
df_wf = df_wf.sort_values(by=["_gen", "_tier", "_seed", "Blotter Name"]).reset_index(
    drop=True
)

# Determine dynamic default target: Latest generation ex_post_blend
blend_rows = df_wf[df_wf["Blotter Name"].str.contains("ex_post_blend", case=False)]
if not blend_rows.empty:
    default_target_name = blend_rows.iloc[-1]["Blotter Name"]
else:
    default_target_name = df_wf.iloc[-1]["Blotter Name"]

blotter_choices = df_wf["Blotter Name"].tolist()

print("\n" + "=" * 105)
print("🏆 CONTINUOUS WALK-FORWARD LEADERBOARD")
print("=" * 105)

summary_cols = [
    "Blotter Name",
    "Total Return",
    "BM Return",
    "Excess Return",
    "Sharpe",
    "Sortino",
    "IR",
    "Max DD",
    "Beta",
    "Sessions",
]
display_df = df_wf[summary_cols].copy()
display_df["Total Return"] = display_df["Total Return"].map(lambda x: f"{x*100:+.2f}%")
display_df["BM Return"] = display_df["BM Return"].map(lambda x: f"{x*100:+.2f}%")
display_df["Excess Return"] = display_df["Excess Return"].map(
    lambda x: f"{x*100:+.2f}%"
)
display_df["Max DD"] = display_df["Max DD"].map(lambda x: f"{x*100:.2f}%")
display_df["Sharpe"] = display_df["Sharpe"].map(lambda x: f"{x:.3f}")
display_df["Sortino"] = display_df["Sortino"].map(lambda x: f"{x:.3f}")
display_df["IR"] = display_df["IR"].map(lambda x: f"{x:.3f}")
display_df["Beta"] = display_df["Beta"].map(lambda x: f"{x:.2f}")

lines = display_df.to_string(index=False).splitlines()
header = lines[0]
data_lines = lines[1:]

output_lines = [header]
prev_gen = None
for gen, line in zip(df_wf["_gen"], data_lines):
    if prev_gen is not None and gen != prev_gen:
        output_lines.append("")  # Visual demarcation between generation families
    output_lines.append(line)
    prev_gen = gen

print("\n".join(output_lines))
print("=" * 105)
print(f"💡 Default Target Detected : {default_target_name}")

🔍 Found 27 Continuous Blotters & 109 Checkpoints.

🏆 CONTINUOUS WALK-FORWARD LEADERBOARD
                                  Blotter Name Total Return BM Return Excess Return Sharpe Sortino     IR  Max DD Beta  Sessions
blotter_continuous_ex_post_blend_gen17.parquet     +129.13%   +76.68%       +52.45%  0.862   0.803  0.537 -23.73% 1.27      1108
          blotter_continuous_s42_gen17.parquet     +139.10%   +76.68%       +62.41%  0.839   0.712  0.503 -26.31% 1.27      1108
         blotter_continuous_s101_gen17.parquet      +68.95%   +76.68%        -7.74%  0.743  -0.214 -0.135 -21.53% 1.00      1108
         blotter_continuous_s777_gen17.parquet     +174.51%   +76.68%       +97.82%  0.831   0.847  0.588 -33.51% 1.52      1108
    blotter_continuous_committee_gen17.parquet      +58.30%   +76.68%       -18.38%  0.596  -0.248 -0.184 -25.78% 1.13      1108

blotter_continuous_ex_post_blend_gen18.parquet     +129.56%   +75.04%       +54.52%  0.865   0.828  0.555 -22.75% 1.26      1110
       

 ### CELL 3A: Interactive Target Selector

 - Select any continuous walk-forward blotter from the dropdown.

 - Alternatively, set `TARGET_OVERRIDE` in code to pin an explicit filename or index.

 - After selecting, execute **Cell 3B onwards** (`Shift + Enter` or click **Run Below**).

In [3]:
# CELL 3A: Interactive Target Selector
TARGET_OVERRIDE: Optional[Union[str, int]] = (
    None  # e.g., "blotter_continuous_s777_gen21.parquet" or None to use Dropdown
)

import ipywidgets as widgets
from IPython.display import display

# Preserve existing widget if already instantiated so selections are never wiped
if "target_selector" not in globals():
    initial_val = (
        default_target_name
        if default_target_name in blotter_choices
        else blotter_choices[-1]
    )
    target_selector = widgets.Dropdown(
        options=blotter_choices,
        value=initial_val,
        description="Target Blotter:",
        layout=widgets.Layout(width="720px"),
    )

    def _on_target_change(change):
        if change["new"]:
            print(
                f"👉 Target set to: {change['new']} — Press Shift+Enter on Cell 3B (or click 'Run Below') to analyze!"
            )

    target_selector.observe(_on_target_change, names="value")

# Programmatic override handling (if set in code)
if TARGET_OVERRIDE is not None:
    if isinstance(TARGET_OVERRIDE, int) and 0 <= TARGET_OVERRIDE < len(blotter_choices):
        target_selector.value = blotter_choices[TARGET_OVERRIDE]
    elif (
        isinstance(TARGET_OVERRIDE, str) and TARGET_OVERRIDE.strip() in blotter_choices
    ):
        target_selector.value = TARGET_OVERRIDE.strip()

display(target_selector)

Dropdown(description='Target Blotter:', index=23, layout=Layout(width='720px'), options=('blotter_continuous_e…

In [4]:
# CELL 3B: Target File Ingestion & Metadata Alignment
target_file = (
    target_selector.value if "target_selector" in globals() else default_target_name
)
print(f"🎯 Target File Active: {target_file}")

# Locate file on disk
target_path = OUTPUT_DIR / target_file
if not target_path.exists():
    target_path = checkpoint_dir / target_file
if not target_path.exists():
    matches = list(OUTPUT_DIR.rglob(target_file))
    if matches:
        target_path = matches[0]

if not target_path.exists():
    raise FileNotFoundError(f"❌ Target file not found: {target_file}")

is_parquet = target_path.suffix == ".parquet"

# Parse Target Generation & Personality Metadata
gen_match = re.search(r"gen(\d+)", target_file, re.IGNORECASE)
target_gen = int(gen_match.group(1)) if gen_match else None
is_blend = "ex_post_blend" in target_file.lower()
seed_match = re.search(r"_s(\d+)(?:_|\.|$)", target_file.lower())
target_seed = int(seed_match.group(1)) if seed_match else None

active_ckpt = None
active_ckpt_path = None

if is_parquet:
    print(f"⚡ Loading Pre-Computed Authoritative Blotter: {target_path.name}")
    blotter_df = pd.read_parquet(target_path)
    date_col = "date" if "date" in blotter_df.columns else "decision_date"
    blotter_df["date"] = pd.to_datetime(blotter_df[date_col])
    blotter_df = blotter_df.sort_values("date").reset_index(drop=True)
else:
    print(
        f"🤖 Checkpoint detected ({target_path.name}). Launching OOS Replay Environment..."
    )
    data = load_processed_data()
    df_close = data.df_ohlcv["Adj Close"].unstack(level=0).sort_index()
    macro_df = data.macro_df
    trading_calendar = get_master_trading_calendar(
        data.df_ohlcv, base_config.calendar_ticker
    )
    feature_cube = pd.read_parquet(LOCAL_DATA_DIR / CacheConfig.get_filename())

    reward_matrix = df_close.pct_change(1, fill_method=None).shift(-1)
    reward_matrix["CASH"] = 0.0

    cal_test = trading_calendar[
        (trading_calendar >= pd.Timestamp("2022-03-25"))
        & (trading_calendar.isin(feature_cube.index.get_level_values("Date").unique()))
    ]

    ckpt = torch.load(target_path, map_location=device, weights_only=False)
    active_ckpt = ckpt
    active_ckpt_path = target_path
    m_state = ckpt.get("model_state_dict", ckpt)
    s_state = ckpt.get("scaler_state", None)

    env_test = DiscoveryEnv(
        feature_cube=feature_cube,
        simple_ret_matrix=reward_matrix,
        calendar=cal_test,
        macro_df=macro_df,
        config=base_config,
        randomize_start=False,
        episode_steps=0,
    )
    gym_test = RLVRGymEnv(env_test, macro_df)
    gym_test.is_training = False
    if s_state is not None:
        gym_test.scaler.load_state(s_state)

    ag = AbsoluteZeroAgent(
        obs_dim=gym_test.observation_space.shape[0],
        action_dim=gym_test.action_space.shape[0],
    ).to(device)
    ag.load_state_dict(m_state)

    eval_res = AgentEvaluator.evaluate(ag, gym_test, device=device, detailed_log=True)
    blotter_df = pd.DataFrame(eval_res["blotter"])
    blotter_df["date"] = pd.to_datetime(blotter_df["date"])
    blotter_df = blotter_df.sort_values("date").reset_index(drop=True)

print(
    f"✅ Successfully ingested {len(blotter_df)} sessions "
    f"({blotter_df['date'].min().strftime('%Y-%m-%d')} to {blotter_df['date'].max().strftime('%Y-%m-%d')})"
)
print(
    f"   • Architecture Type : {'Ex-Post Capital Blend' if is_blend else (f'Constituent Seed {target_seed}' if target_seed else 'Custom Replay')}"
)
print(
    f"   • Generation Family : Gen {target_gen if target_gen is not None else 'Unspecified'}"
)

🎯 Target File Active: blotter_continuous_ex_post_blend_gen22.parquet
⚡ Loading Pre-Computed Authoritative Blotter: blotter_continuous_ex_post_blend_gen22.parquet
✅ Successfully ingested 1110 sessions (2022-03-25 to 2026-09-04)
   • Architecture Type : Ex-Post Capital Blend
   • Generation Family : Gen 22


In [5]:
# CELL 4: Dynamic Action Space Extraction & Factor Alignment
registry = get_strategy_registry(base_config)
factor_names = list(registry.keys())
action_names = factor_names + [
    "Rank Offset",
    "Rank Width",
    "Equity Exposure",
    "Active Tilt",
]

# If raw_actions exist directly in blotter, decode them
if "raw_actions" in blotter_df.columns:
    raw_acts = np.vstack(blotter_df["raw_actions"].values)
    if raw_acts.shape[1] == len(action_names):
        for idx, col_name in enumerate(action_names):
            blotter_df[col_name] = raw_acts[:, idx]
        print(
            f"✅ Decoded {len(action_names)} raw action dimensions directly from blotter."
        )
elif is_blend and target_gen is not None:
    # Dynamically locate all constituent seeds for THIS specific generation
    candidate_dirs = [OUTPUT_DIR, OUTPUT_DIR / "canonical_anchors"]
    c_files = []
    for s_dir in candidate_dirs:
        if s_dir.exists():
            c_files.extend(
                list(s_dir.glob(f"blotter_continuous_s*_gen{target_gen}.parquet"))
            )

    # Deduplicate matching files
    seen = set()
    constituent_files = []
    for f in sorted(c_files):
        if f.name not in seen:
            seen.add(f.name)
            constituent_files.append(f)

    if constituent_files:
        print(
            f"⚡ Synthesizing average action exposure across {len(constituent_files)} constituent seeds for Gen {target_gen} blend..."
        )
        c_dfs = [pd.read_parquet(f) for f in constituent_files]
        if "raw_actions" in c_dfs[0].columns:
            avg_acts = np.mean(
                [
                    np.vstack(df["raw_actions"].values)
                    for df in c_dfs
                    if "raw_actions" in df.columns
                ],
                axis=0,
            )
            if avg_acts.shape[1] == len(action_names):
                for idx, col_name in enumerate(action_names):
                    blotter_df[col_name] = avg_acts[:, idx]
                seed_labels = [
                    re.search(r"_s(\d+)_", f.name).group(1)
                    for f in constituent_files
                    if re.search(r"_s(\d+)_", f.name)
                ]
                print(
                    f"✅ Synthetic ensemble action space mapped successfully from seeds {seed_labels}."
                )
    else:
        print(
            f"ℹ️ No peer constituent seed blotters found for Gen {target_gen}. Skipping action synthesis."
        )

⚡ Synthesizing average action exposure across 3 constituent seeds for Gen 22 blend...
✅ Synthetic ensemble action space mapped successfully from seeds ['101', '42', '777'].


In [6]:
# CELL 5: Institutional Regime Attribution Analysis
REGIMES = [
    {
        "name": "Regime 1: Fed Rate Shock (2022)",
        "start": "2022-03-25",
        "end": "2022-12-30",
        "desc": "Aggressive tightening, multiple 75bps rate hikes, severe tech multiple compression.",
    },
    {
        "name": "Regime 2: Tech Rebound & AI Expansion (2023)",
        "start": "2023-01-03",
        "end": "2023-12-29",
        "desc": "Mega-cap tech leadership, LLM inflection, market recovery.",
    },
    {
        "name": "Regime 3: Macro Dispersion & Fed Pivot (2024-2026)",
        "start": "2024-01-02",
        "end": "2026-09-02",
        "desc": "Broad bull rally, rate cut cycle inception, high cross-sectional dispersion.",
    },
]


def audit_regime_slice(df_slice: pd.DataFrame, regime_name: str) -> Dict[str, Any]:
    net_r = df_slice["net_daily_simple_ret"].to_numpy(dtype=float)
    bm_r = df_slice["bm_daily_simple_ret"].to_numpy(dtype=float)
    sp_r = net_r - bm_r

    cum_strat = float(np.prod(1.0 + net_r) - 1.0)
    cum_bm = float(np.prod(1.0 + bm_r) - 1.0)
    excess = cum_strat - cum_bm

    mean_r = float(np.mean(net_r))
    std_r = float(np.std(net_r, ddof=1)) if len(net_r) > 1 else 1e-8
    sharpe = float((mean_r / std_r) * np.sqrt(252)) if std_r > 1e-8 else 0.0

    mean_sp = float(np.mean(sp_r))
    std_sp = float(np.std(sp_r, ddof=1)) if len(sp_r) > 1 else 1e-8
    ir = float((mean_sp / std_sp) * np.sqrt(252)) if std_sp > 1e-8 else 0.0

    downside = sp_r[sp_r < 0]
    downside_std = float(np.std(downside, ddof=1)) if len(downside) > 1 else 1e-8
    sortino = (
        float((mean_sp / downside_std) * np.sqrt(252)) if downside_std > 1e-8 else 0.0
    )

    bm_var = float(np.var(bm_r, ddof=1)) if len(bm_r) > 1 else 1e-8
    beta = float(np.cov(net_r, bm_r)[0, 1] / bm_var) if bm_var > 1e-8 else 1.0

    eq = np.cumprod(1.0 + net_r)
    peaks = np.maximum.accumulate(eq)
    dds = (eq - peaks) / np.maximum(peaks, 1e-8)
    max_dd = float(np.min(dds))

    return {
        "Regime": regime_name,
        "Start Date": df_slice["date"].iloc[0].strftime("%Y-%m-%d"),
        "End Date": df_slice["date"].iloc[-1].strftime("%Y-%m-%d"),
        "Sessions": len(df_slice),
        "Strategy Return": cum_strat,
        "BM Return": cum_bm,
        "Excess Spread": excess,
        "Sharpe": sharpe,
        "Sortino": sortino,
        "IR": ir,
        "Beta": beta,
        "Max DD": max_dd,
    }


regime_records = []
for reg in REGIMES:
    sub_df = blotter_df[
        (blotter_df["date"] >= pd.Timestamp(reg["start"]))
        & (blotter_df["date"] <= pd.Timestamp(reg["end"]))
    ]
    if not sub_df.empty:
        regime_records.append(audit_regime_slice(sub_df, reg["name"]))

# Full Period Aggregation
regime_records.append(
    audit_regime_slice(blotter_df, "FULL PERIOD (Continuous Trajectory)")
)
reg_df = pd.DataFrame(regime_records)

print("\n" + "=" * 115)
print(f"🏛️ INSTITUTIONAL REGIME-PARTITIONED AUDIT — {target_file}")
print("=" * 115)
reg_display = reg_df.copy()
reg_display["Strategy Return"] = reg_display["Strategy Return"].map(
    lambda x: f"{x*100:+.2f}%"
)
reg_display["BM Return"] = reg_display["BM Return"].map(lambda x: f"{x*100:+.2f}%")
reg_display["Excess Spread"] = reg_display["Excess Spread"].map(
    lambda x: f"{x*100:+.2f}%"
)
reg_display["Sharpe"] = reg_display["Sharpe"].map(lambda x: f"{x:.3f}")
reg_display["Sortino"] = reg_display["Sortino"].map(lambda x: f"{x:.3f}")
reg_display["IR"] = reg_display["IR"].map(lambda x: f"{x:.3f}")
reg_display["Beta"] = reg_display["Beta"].map(lambda x: f"{x:.2f}")
reg_display["Max DD"] = reg_display["Max DD"].map(lambda x: f"{x*100:.2f}%")
print(
    reg_display[
        [
            "Regime",
            "Sessions",
            "Strategy Return",
            "BM Return",
            "Excess Spread",
            "Sharpe",
            "IR",
            "Beta",
            "Max DD",
        ]
    ].to_string(index=False)
)
print("=" * 115)


🏛️ INSTITUTIONAL REGIME-PARTITIONED AUDIT — blotter_continuous_ex_post_blend_gen22.parquet
                                            Regime  Sessions Strategy Return BM Return Excess Spread Sharpe     IR Beta  Max DD
                   Regime 1: Fed Rate Shock (2022)       194          -7.93%   -14.78%        +6.84%  0.054  0.577 1.69 -29.19%
      Regime 2: Tech Rebound & AI Expansion (2023)       248         +21.48%   +24.95%        -3.47%  1.321 -0.344 1.09 -11.97%
Regime 3: Macro Dispersion & Fed Pivot (2024-2026)       666         +71.92%   +65.93%        +5.99%  1.209  0.244 1.07 -21.58%
               FULL PERIOD (Continuous Trajectory)      1110         +90.73%   +75.04%       +15.69%  0.675  0.255 1.30 -29.19%


In [7]:
# CELL 6: Forensic Beta & Alpha Decomposition (CAPM Attribution)
print("\n" + "=" * 80)
print(f"🔬 FORENSIC BETA DECOMPOSITION & CAPM ATTRIBUTION ({target_file})")
print("=" * 80)

r_p = blotter_df["net_daily_simple_ret"].to_numpy(dtype=float)
r_bm = blotter_df["bm_daily_simple_ret"].to_numpy(dtype=float)

cov_mat = np.cov(r_p, r_bm)
var_bm = cov_mat[1, 1]
realized_beta = cov_mat[0, 1] / var_bm
mean_p_daily = np.mean(r_p)
mean_bm_daily = np.mean(r_bm)

daily_alpha = mean_p_daily - realized_beta * mean_bm_daily
ann_alpha = daily_alpha * 252

residuals = r_p - (realized_beta * r_bm)
idiosyncratic_vol = np.std(residuals, ddof=1) * np.sqrt(252)
total_vol = np.std(r_p, ddof=1) * np.sqrt(252)
bm_vol = np.std(r_bm, ddof=1) * np.sqrt(252)
r_squared = (cov_mat[0, 1] ** 2) / (cov_mat[0, 0] * cov_mat[1, 1])

bm_cum = np.prod(1.0 + r_bm) - 1.0
pure_leveraged_bm_ret = np.prod(1.0 + realized_beta * r_bm) - 1.0
actual_strategy_ret = np.prod(1.0 + r_p) - 1.0
alpha_over_leveraged_bm = actual_strategy_ret - pure_leveraged_bm_ret

print(f"  • Realized CAPM Beta to {benchmark_name}       : {realized_beta:.4f}")
print(f"  • Regression R-Squared (Market Fit)   : {r_squared*100:.2f}%")
print(
    f"  • Idiosyncratic Volatility (Non-Market): {idiosyncratic_vol*100:.2f}% ({(idiosyncratic_vol/total_vol)*100:.1f}% of total risk)"
)
print(f"  • Annualized CAPM Alpha (Pure Alpha)  : {ann_alpha*100:+.2f}%")
print("-" * 80)
print(f"  • Raw Benchmark Cumulative Return     : {bm_cum*100:+.2f}%")
print(
    f"  • Synthetic Leveraged Benchmark ({realized_beta:.2f}x): {pure_leveraged_bm_ret*100:+.2f}%"
)
print(f"  • Actual Realized Strategy Return     : {actual_strategy_ret*100:+.2f}%")
print(f"  • 🔥 PURE ALPHA SPREAD OVER LEVERAGE : {alpha_over_leveraged_bm*100:+.2f}%")
print("=" * 80)

# Rolling 63-Day Systematic Beta & Alpha Plot
window = 63
rolling_cov = pd.Series(r_p).rolling(window).cov(pd.Series(r_bm)).to_numpy()
rolling_var = pd.Series(r_bm).rolling(window).var().to_numpy()
rolling_beta = rolling_cov / np.maximum(rolling_var, 1e-8)

rolling_p_ret = pd.Series(r_p).rolling(window).mean() * 252
rolling_bm_ret = pd.Series(r_bm).rolling(window).mean() * 252
rolling_alpha = rolling_p_ret - (rolling_beta * rolling_bm_ret)

fig_beta = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.08,
    subplot_titles=(
        f"🌊 63-Day Rolling Systematic Beta to {benchmark_name}",
        "⚡ 63-Day Rolling Annualized CAPM Alpha (%)",
    ),
)
fig_beta.add_trace(
    go.Scatter(
        x=blotter_df["date"],
        y=rolling_beta,
        mode="lines",
        name="Rolling 63d Beta",
        line=dict(color="#636EFA", width=1.8),
    ),
    row=1,
    col=1,
)
fig_beta.add_hline(
    y=realized_beta,
    line_dash="dash",
    line_color="black",
    annotation_text=f"Full Beta: {realized_beta:.2f}",
    row=1,
    col=1,
)
fig_beta.add_hline(y=1.0, line_dash="dot", line_color="gray", row=1, col=1)

fig_beta.add_trace(
    go.Scatter(
        x=blotter_df["date"],
        y=rolling_alpha * 100,
        mode="lines",
        name="Rolling Alpha (%)",
        line=dict(color="#00CC96", width=1.8),
    ),
    row=2,
    col=1,
)
fig_beta.add_hline(y=0.0, line_dash="dot", line_color="crimson", row=2, col=1)
fig_beta.update_layout(
    height=600,
    width=1100,
    hovermode="x unified",
    title_text=f"Dynamic Factor Exposure: Rolling Beta & CAPM Alpha Dynamics ({target_file})",
)
fig_beta.show()


🔬 FORENSIC BETA DECOMPOSITION & CAPM ATTRIBUTION (blotter_continuous_ex_post_blend_gen22.parquet)
  • Realized CAPM Beta to SPY       : 1.3014
  • Regression R-Squared (Market Fit)   : 67.86%
  • Idiosyncratic Volatility (Non-Market): 15.38% (56.7% of total risk)
  • Annualized CAPM Alpha (Pure Alpha)  : -0.14%
--------------------------------------------------------------------------------
  • Raw Benchmark Cumulative Return     : +75.04%
  • Synthetic Leveraged Benchmark (1.30x): +102.02%
  • Actual Realized Strategy Return     : +90.73%
  • 🔥 PURE ALPHA SPREAD OVER LEVERAGE : -11.29%


In [8]:
# CELL 7: Continuous Trajectory, Multi-Asset Breakdown & Drawdown Timeline
eq_curve = blotter_df["agent_equity"].to_numpy(dtype=float)
bm_curve = blotter_df["bm_equity"].to_numpy(dtype=float)
peaks = np.maximum.accumulate(eq_curve)
dds = (eq_curve - peaks) / np.maximum(peaks, 1e-8)
blotter_df["Drawdown"] = dds

has_alloc = "weight_active" in blotter_df.columns

fig = make_subplots(
    rows=4,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.04,
    row_heights=[0.38, 0.22, 0.20, 0.20],
    subplot_titles=(
        f"📈 Realized Trajectory vs {benchmark_name}",
        "🔥 Cumulative Alpha Spread Multiplier (V_p / V_bm)",
        f"🏛️ Dynamic Tri-Asset Allocation (Stocks / {benchmark_name} / Cash)",
        "📉 Drawdown Profile",
    ),
)

fig.add_trace(
    go.Scatter(
        x=blotter_df["date"],
        y=blotter_df["agent_equity"],
        mode="lines",
        name="Strategy",
        line=dict(color="#00CC96", width=2.5),
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=blotter_df["date"],
        y=blotter_df["bm_equity"],
        mode="lines",
        name=benchmark_name,
        line=dict(color="#111827", width=2, dash="dash"),
    ),
    row=1,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=blotter_df["date"],
        y=blotter_df["alpha_equity"],
        mode="lines",
        name="Alpha Equity",
        line=dict(color="#636EFA", width=2.0),
    ),
    row=2,
    col=1,
)
fig.add_hline(y=1.0, line_dash="dot", line_color="gray", row=2, col=1)

if has_alloc:
    fig.add_trace(
        go.Scatter(
            x=blotter_df["date"],
            y=blotter_df["weight_active"],
            mode="lines",
            stackgroup="alloc",
            name="Active Stocks",
            line=dict(color="forestgreen", width=0.5),
        ),
        row=3,
        col=1,
    )
    fig.add_trace(
        go.Scatter(
            x=blotter_df["date"],
            y=blotter_df["weight_benchmark"],
            mode="lines",
            stackgroup="alloc",
            name=f"Shelter ({benchmark_name})",
            line=dict(color="royalblue", width=0.5),
        ),
        row=3,
        col=1,
    )
    fig.add_trace(
        go.Scatter(
            x=blotter_df["date"],
            y=blotter_df["weight_cash"],
            mode="lines",
            stackgroup="alloc",
            name="Cash",
            line=dict(color="gold", width=0.5),
        ),
        row=3,
        col=1,
    )

fig.add_trace(
    go.Scatter(
        x=blotter_df["date"],
        y=blotter_df["Drawdown"],
        mode="lines",
        fill="tozeroy",
        name="Drawdown",
        line=dict(color="crimson", width=1.0),
    ),
    row=4,
    col=1,
)

for reg in REGIMES[:-1]:
    fig.add_vline(x=reg["end"], line_dash="dash", line_color="gray", row=1, col=1)
    fig.add_vline(x=reg["end"], line_dash="dash", line_color="gray", row=2, col=1)

fig.update_layout(
    height=1000,
    width=1150,
    hovermode="x unified",
    title_text=f"Continuous Trajectory & Multi-Asset Breakdown ({target_file})",
)
fig.update_yaxes(title_text="Multiplier", row=1, col=1)
fig.update_yaxes(title_text="Alpha (V_p/V_bm)", row=2, col=1)
fig.update_yaxes(
    title_text="Exposure %", tickformat=".0%", range=[0, 1.0], row=3, col=1
)
fig.update_yaxes(title_text="Drawdown %", tickformat=".1%", row=4, col=1)
fig.show()

In [9]:
# CELL 8: Agent Personality & Factor Tilts Over Time
if factor_names[0] in blotter_df.columns:
    mean_factors = blotter_df[factor_names].mean().reset_index()
    mean_factors.columns = ["Factor", "Average Weight"]
    mean_factors = mean_factors.sort_values("Average Weight", ascending=True)

    fig_personality = px.bar(
        mean_factors,
        x="Average Weight",
        y="Factor",
        orientation="h",
        title=f"🤖 Strategy Personality & Average Factor Tilts — {target_file}",
        color="Average Weight",
        color_continuous_scale="Viridis",
    )
    fig_personality.update_layout(height=500, width=950)
    fig_personality.show()

    # Rolling 63-Day Factor Dynamics (Top 4 needle-moving factors)
    top_4_factors = mean_factors.tail(4)["Factor"].tolist()
    fig_rolling_f = go.Figure()
    for factor in top_4_factors:
        fig_rolling_f.add_trace(
            go.Scatter(
                x=blotter_df["date"],
                y=blotter_df[factor].rolling(63).mean(),
                mode="lines",
                name=factor,
            )
        )
    fig_rolling_f.update_layout(
        title=f"🌊 63-Day Rolling Factor Scoring Weights (Top Needles — {target_file})",
        xaxis_title="Date",
        yaxis_title="Smoothed Alpha Weight",
        height=450,
        width=1100,
    )
    fig_rolling_f.show()
else:
    print(f"ℹ️ Factor columns not present in {target_file}. Personality radar skipped.")

In [10]:
# CELL 9: Granular Blotter & Trade Inspector (Single Session Transparency)
sample_idx = -1
row = blotter_df.iloc[sample_idx]

date_str = (
    pd.to_datetime(row["date"]).strftime("%Y-%m-%d") if pd.notna(row["date"]) else "N/A"
)
buy_str = (
    pd.to_datetime(row.get("buy_date")).strftime("%Y-%m-%d")
    if pd.notna(row.get("buy_date", None))
    else "N/A"
)
sell_str = (
    pd.to_datetime(row.get("sell_date")).strftime("%Y-%m-%d")
    if pd.notna(row.get("sell_date", None))
    else "N/A"
)

print("=" * 75)
print(f"📋 GRANULAR BLOTTER INSPECTOR — Decision Date: {date_str} ({target_file})")
print("=" * 75)

print("🗓️  1. TRADE TIMELINE & EXECUTION SPECS")
print(f"  • Decision Date (T)     : {date_str}")
print(f"  • Execution Buy Date    : {buy_str}")
print(f"  • Liquidation Date      : {sell_str}")
print(f"  • Holding Period Buffer : {base_config.holding_period} trading sessions")
print("-" * 75)

print("🏛️  2. CAPITAL PARTITIONING & TRI-ASSET DIALS")
print(
    f"  • Active Stock Weight   : {row.get('weight_active', 1.0)*100:.2f}% (Cross-Sectional Basket)"
)
print(
    f"  • Benchmark Weight      : {row.get('weight_benchmark', 0.0)*100:.2f}% ({benchmark_name} Shelter)"
)
print(
    f"  • Cash Weight           : {row.get('weight_cash', 0.0)*100:.2f}% (Risk-Free Reserve)"
)
print(
    f"  • Decoded Basket Width  : {row.get('width', row.get('Rank Width', 'N/A'))} tickers"
)
print(f"  • Decoded Rank Offset   : {row.get('offset', row.get('Rank Offset', 'N/A'))}")
print("-" * 75)

if "tickers" in row and isinstance(row["tickers"], (list, np.ndarray)):
    print(f"📦  3. SELECTED BASKET ASSETS ({len(row['tickers'])} tickers):")
    for rank_i, sym in enumerate(row["tickers"], start=1):
        print(
            f"    [{rank_i:02d}] Ticker: {sym:<6} (Uniform intra-sleeve weight: {100.0/len(row['tickers']):.2f}%)"
        )
    print("-" * 75)

print("💰  4. REALIZED DAILY MTM RETURN ACCOUNTING")
r_stock = row.get("gross_stock_daily_simple_ret", np.nan)
r_mkt = row.get("bm_daily_simple_ret", 0.0)
r_net = row.get("net_daily_simple_ret", 0.0)
r_alpha = row.get("alpha_daily_simple_ret", 0.0)
slippage = row.get("slippage_daily_simple_loss", 0.0)

if pd.notna(r_stock):
    print(f"  • Active Basket Return  : {r_stock*100:+.3f}%")
print(f"  • Benchmark Return      : {r_mkt*100:+.3f}% ({benchmark_name})")
print(f"  • Amortized Slippage    : -{slippage*100:.4f}% ({slippage*10000:.1f} bps)")
print(f"  • Net Strategy Return   : {r_net*100:+.3f}%")
print(f"  • Net Daily Alpha Spread: {r_alpha*100:+.3f}%")
print(f"  • Portfolio Multiplier  : {row.get('agent_equity', 1.0):.4f}")
print(f"  • Alpha Multiplier      : {row.get('alpha_equity', 1.0):.4f}")
print("=" * 75)

📋 GRANULAR BLOTTER INSPECTOR — Decision Date: 2026-09-04 (blotter_continuous_ex_post_blend_gen22.parquet)
🗓️  1. TRADE TIMELINE & EXECUTION SPECS
  • Decision Date (T)     : 2026-09-04
  • Execution Buy Date    : N/A
  • Liquidation Date      : N/A
  • Holding Period Buffer : 5 trading sessions
---------------------------------------------------------------------------
🏛️  2. CAPITAL PARTITIONING & TRI-ASSET DIALS
  • Active Stock Weight   : 100.00% (Cross-Sectional Basket)
  • Benchmark Weight      : 0.00% (SPY Shelter)
  • Cash Weight           : 0.00% (Risk-Free Reserve)
  • Decoded Basket Width  : -0.021116132537523907 tickers
  • Decoded Rank Offset   : -0.44456638892491657
---------------------------------------------------------------------------
💰  4. REALIZED DAILY MTM RETURN ACCOUNTING
  • Benchmark Return      : -0.549% (SPY)
  • Amortized Slippage    : -0.0000% (0.0 bps)
  • Net Strategy Return   : -0.570%
  • Net Daily Alpha Spread: -0.020%
  • Portfolio Multiplier  : 1.90

In [11]:
# CELL 10: Extreme Tail Event Radar (Best Sessions vs Crash Days)
print(f"\n🌟 TOP 5 OUTPERFORMANCE SESSIONS ({target_file}):")
top5 = blotter_df.nlargest(5, "alpha_daily_simple_ret")[
    [
        "date",
        "net_daily_simple_ret",
        "bm_daily_simple_ret",
        "alpha_daily_simple_ret",
        "agent_equity",
    ]
].copy()
top5["date"] = top5["date"].dt.strftime("%Y-%m-%d")
for c in ["net_daily_simple_ret", "bm_daily_simple_ret", "alpha_daily_simple_ret"]:
    top5[c] = top5[c].apply(lambda x: f"{x*100:+.2f}%")
print(top5.to_string(index=False))

print(f"\n🚨 TOP 5 UNDERPERFORMANCE (CRASH) SESSIONS ({target_file}):")
worst5 = blotter_df.nsmallest(5, "alpha_daily_simple_ret")[
    [
        "date",
        "net_daily_simple_ret",
        "bm_daily_simple_ret",
        "alpha_daily_simple_ret",
        "agent_equity",
    ]
].copy()
worst5["date"] = worst5["date"].dt.strftime("%Y-%m-%d")
for c in ["net_daily_simple_ret", "bm_daily_simple_ret", "alpha_daily_simple_ret"]:
    worst5[c] = worst5[c].apply(lambda x: f"{x*100:+.2f}%")
print(worst5.to_string(index=False))

if factor_names[0] in blotter_df.columns:
    worst_idx = blotter_df["net_daily_simple_ret"].idxmin()
    worst_row = blotter_df.iloc[worst_idx]
    worst_date_str = worst_row["date"].strftime("%Y-%m-%d")

    crash_data = pd.DataFrame(
        {
            "Factor": factor_names,
            "Crash Weight": worst_row[factor_names].values.astype(float),
            "Average Weight": blotter_df[factor_names].mean().values.astype(float),
        }
    )

    fig_radar = px.line_polar(
        crash_data,
        r="Crash Weight",
        theta="Factor",
        line_close=True,
        title=f"🚨 Factor Tilt Radar on Worst Strategy Day ({worst_date_str}) vs Mean Tilt ({target_file})",
    )
    fig_radar.add_trace(
        go.Scatterpolar(
            r=crash_data["Average Weight"],
            theta=crash_data["Factor"],
            name="Mean Profile",
            line=dict(color="blue", dash="dot"),
        )
    )
    fig_radar.update_traces(fill="toself")
    fig_radar.show()


🌟 TOP 5 OUTPERFORMANCE SESSIONS (blotter_continuous_ex_post_blend_gen22.parquet):
      date net_daily_simple_ret bm_daily_simple_ret alpha_daily_simple_ret  agent_equity
2022-07-01               +9.40%              +0.19%                 +9.21%      0.907484
2022-05-12              +10.63%              +2.39%                 +8.24%      0.852522
2022-07-19               +8.29%              +0.64%                 +7.65%      1.064944
2022-05-11               +6.60%              -0.10%                 +6.70%      0.770585
2022-07-06               +7.52%              +1.50%                 +6.02%      0.954120

🚨 TOP 5 UNDERPERFORMANCE (CRASH) SESSIONS (blotter_continuous_ex_post_blend_gen22.parquet):
      date net_daily_simple_ret bm_daily_simple_ret alpha_daily_simple_ret  agent_equity
2022-07-21               -7.11%              -0.93%                 -6.18%      0.995174
2022-05-23               -6.72%              -0.76%                 -5.95%      0.774735
2022-05-10             

In [12]:
# CELL 11: Multi-Seed Arithmetic & Compounding Auditor
def audit_session_transactions(sample_idx: int = 0):
    """Audits constituent seed transactions dynamically for the target's generation family
    and mathematically verifies the ex-post blended return and step compounding.
    """
    gen_to_audit = target_gen if target_gen is not None else 18
    blend_name = f"blotter_continuous_ex_post_blend_gen{gen_to_audit}.parquet"

    # Search for constituent blotters
    candidate_dirs = [OUTPUT_DIR, OUTPUT_DIR / "canonical_anchors"]
    c_map = {}
    blend_path_found = None

    for d in candidate_dirs:
        if not d.exists():
            continue
        if (d / blend_name).exists() and blend_path_found is None:
            blend_path_found = d / blend_name
        for p in d.glob(f"blotter_continuous_s*_gen{gen_to_audit}.parquet"):
            sm = re.search(r"_s(\d+)_", p.name)
            if sm:
                seed_id = int(sm.group(1))
                if seed_id not in c_map:
                    c_map[seed_id] = p

    if not blend_path_found or not c_map:
        print(
            f"ℹ️ Peer audit files for Gen {gen_to_audit} not fully located. Performing single-target compounding audit."
        )
        df_target = blotter_df
        idx = sample_idx if sample_idx >= 0 else len(df_target) + sample_idx
        row = df_target.iloc[idx]
        prev_row = df_target.iloc[idx - 1] if idx > 0 else None
        prev_eq = prev_row["agent_equity"] if prev_row is not None else 1.0
        r_net = row["net_daily_simple_ret"]
        calc_eq = prev_eq * (1.0 + r_net)
        pub_eq = row["agent_equity"]
        assert (
            abs(calc_eq - pub_eq) < 1e-6
        ), f"Equity step compounding mismatch: {calc_eq} vs {pub_eq}"
        print(
            f"✅ [PASS] Step compounding perfectly verified: {prev_eq:.6f} * (1 + {r_net:+.6f}) == {pub_eq:.6f}"
        )
        return

    constituent_dfs = {}
    for s, p in sorted(c_map.items()):
        df_s = pd.read_parquet(p)
        df_s["date"] = pd.to_datetime(df_s.get("date", df_s.get("decision_date")))
        constituent_dfs[s] = df_s.sort_values("date").reset_index(drop=True)

    df_blend = pd.read_parquet(blend_path_found)
    df_blend["date"] = pd.to_datetime(
        df_blend.get("date", df_blend.get("decision_date"))
    )
    df_blend = df_blend.sort_values("date").reset_index(drop=True)

    n_sessions = len(df_blend)
    idx = sample_idx if sample_idx >= 0 else n_sessions + sample_idx
    row_blend = df_blend.iloc[idx]
    prev_blend = df_blend.iloc[idx - 1] if idx > 0 else None

    date_str = row_blend["date"].strftime("%Y-%m-%d")
    bm_ret = row_blend["bm_daily_simple_ret"]
    r_blend = row_blend["net_daily_simple_ret"]
    eq_blend = row_blend["agent_equity"]
    bm_eq = row_blend["bm_equity"]
    alpha_eq = row_blend["alpha_equity"]

    print("=" * 85)
    print(
        f"📋 CONSTITUENT TRANSACTION & EQUITY AUDIT — Gen {gen_to_audit} | Session [{idx + 1}/{n_sessions}] : {date_str}"
    )
    print("=" * 85)
    print(
        f"🏛️ Benchmark Return ({benchmark_name}) : {bm_ret*100:+.3f}% | Benchmark Equity Multiplier: {bm_eq:.4f}"
    )
    print("-" * 85)

    seed_returns = []
    print("📦 CONSTITUENT SEED EXECUTIONS & BASKET ALLOCATIONS:")

    for s, df_s in constituent_dfs.items():
        row_s = df_s.iloc[idx]
        w_act = row_s.get("weight_active", 1.0)
        w_bm = row_s.get("weight_benchmark", 0.0)
        w_csh = row_s.get("weight_cash", 0.0)
        tickers = row_s.get("tickers", [])
        n_tickers = len(tickers) if isinstance(tickers, (list, np.ndarray)) else 0
        intra_weight = (100.0 / n_tickers) if n_tickers > 0 else 0.0

        r_stock = row_s.get("gross_stock_daily_simple_ret", np.nan)
        slip = row_s.get("slippage_daily_simple_loss", 0.0)
        r_net_s = row_s["net_daily_simple_ret"]
        seed_returns.append(r_net_s)

        print(f"\n  🔹 [SEED {s:>4}]")
        print(
            f"     • Capital Allocation : Active = {w_act*100:.1f}% | Benchmark Shelter = {w_bm*100:.1f}% | Cash = {w_csh*100:.1f}%"
        )
        print(
            f"     • Selected Tickers   : ({n_tickers} stocks @ {intra_weight:.2f}% uniform intra-sleeve weight)"
        )
        if n_tickers > 0:
            ticker_str = ", ".join([f"{t}" for t in tickers[:10]])
            if n_tickers > 10:
                ticker_str += f", ... (+{n_tickers-10} more)"
            print(f"       -> {ticker_str}")
        if pd.notna(r_stock):
            print(f"     • Gross Stock Return : {r_stock*100:+.3f}%")
        print(f"     • Amortized Slippage : -{slip*100:.4f}% ({slip*10000:.1f} bps)")
        print(f"     • Net Realized Return: {r_net_s*100:+.3f}%")
        print(f"     • Seed Equity Curve  : {row_s.get('agent_equity', 1.0):.4f}")

    # Mathematical Proof & Verification
    calc_blend_ret = float(np.mean(seed_returns))
    prev_eq = prev_blend["agent_equity"] if prev_blend is not None else 1.0
    calc_new_eq = prev_eq * (1.0 + r_blend)

    print("\n" + "=" * 85)
    print(
        f"⚖️ EX-POST CAPITAL BLEND RETURN & EQUITY ADVANCEMENT VERIFICATION (Gen {gen_to_audit}):"
    )
    print("=" * 85)
    print(
        f"  • Constituent Net Returns : "
        + ", ".join(
            [
                f"s{s} = {r*100:+.3f}%"
                for s, r in zip(constituent_dfs.keys(), seed_returns)
            ]
        )
    )
    print(
        f"  • Arithmetic Mean Return  : (1/{len(seed_returns)}) * sum(returns) = {calc_blend_ret*100:+.3f}%"
    )
    print(f"  • Realized Blend Return   : {r_blend*100:+.3f}%")
    assert (
        abs(calc_blend_ret - r_blend) < 1e-7
    ), f"Blend return mismatch: {calc_blend_ret} vs {r_blend}"
    print(
        f"  ✅ [MATCH] Linear blend return perfectly matches constituent arithmetic mean."
    )

    print(f"\n  • Prior Equity V_p(t-1)   : {prev_eq:.6f}")
    print(
        f"  • Step Compounding Formula: {prev_eq:.6f} * (1 + {r_blend:+.6f}) = {calc_new_eq:.6f}"
    )
    print(f"  • Published Equity V_p(t) : {eq_blend:.6f}")
    assert (
        abs(calc_new_eq - eq_blend) < 1e-6
    ), f"Equity compounding mismatch: {calc_new_eq} vs {eq_blend}"
    print(f"  ✅ [MATCH] Portfolio wealth advances with zero compounding leakage.")

    print(f"\n  • Benchmark Wealth V_bm(t): {bm_eq:.6f}")
    print(
        f"  • Alpha Ratio V_p / V_bm  : {eq_blend:.6f} / {bm_eq:.6f} = {alpha_eq:.6f}"
    )
    print("=" * 85)


# Execute Verification for Launch Day (Session 0) and Terminal Day (Session -1)
audit_session_transactions(sample_idx=0)
audit_session_transactions(sample_idx=1)
audit_session_transactions(sample_idx=2)
audit_session_transactions(sample_idx=3)
audit_session_transactions(sample_idx=-1)

📋 CONSTITUENT TRANSACTION & EQUITY AUDIT — Gen 22 | Session [1/1110] : 2022-03-25
🏛️ Benchmark Return (SPY) : +0.711% | Benchmark Equity Multiplier: 1.0071
-------------------------------------------------------------------------------------
📦 CONSTITUENT SEED EXECUTIONS & BASKET ALLOCATIONS:

  🔹 [SEED   42]
     • Capital Allocation : Active = 93.2% | Benchmark Shelter = 6.8% | Cash = 0.0%
     • Selected Tickers   : (7 stocks @ 14.29% uniform intra-sleeve weight)
       -> BA, ZTO, NWSA, IDXX, VFC, FISV, BKNG
     • Gross Stock Return : +0.711%
     • Amortized Slippage : -0.0000% (0.0 bps)
     • Net Realized Return: +0.711%
     • Seed Equity Curve  : 1.0071

  🔹 [SEED  101]
     • Capital Allocation : Active = 87.9% | Benchmark Shelter = 12.1% | Cash = 0.0%
     • Selected Tickers   : (7 stocks @ 14.29% uniform intra-sleeve weight)
       -> VB, FFIV, LUV, VNQ, IT, ESGU, SBAC
     • Gross Stock Return : +0.711%
     • Amortized Slippage : -0.0000% (0.0 bps)
     • Net Realized Re

 ### CELL 12: Agent Training Diagnostics & PPO Convergence Telemetry (217_diag)

 - **Diagnostic Forensics**: Reconstructs the 6-panel convergence telemetry directly from policy training logs or PyTorch checkpoints.

 - **Panels Monitored**:

   1. *Total & Policy Loss*: Verifies actor gradient trajectory and total loss stabilization.

   2. *Value Loss (Critic)*: Tracks MSE/Huber critic loss decay and baseline convergence.

   3. *Entropy (Exploration)*: Audits exploration budget decay vs premature mode collapse.

   4. *PPO Stability (Approx KL & Clip Frac)*: Proves updates remain within the clipping trust region.

   5. *Explained Variance ($R^2_{\text{critic}}$)*: Benchmark lines at 1.0 (ideal) and 0.0 (uninformative) to detect critic failure.

   6. *Average Step Reward*: Monitors per-step reward progression across training epochs.

 - **Smart Fallback Discovery**: Automatically links policy checkpoints to embedded or companion telemetry logs, ensuring telemetry plots are always rendered.

In [13]:
# CELL 12: Agent Training Diagnostics & PPO Convergence Telemetry (217_diag)
DIAG_CHECKPOINT_OVERRIDE: Optional[str] = (
    None  # e.g., "model_P9_Scale25_Gamma88_LossAv035_Up118_Tilt10_Width5_s42_ep_78.pt"
)


def _extract_telemetry_df(
    ckpt_obj: Any, candidate_path: Optional[Path] = None
) -> Optional[pd.DataFrame]:
    """Inspects checkpoint dictionaries, nested dicts, and adjacent companion logs

    to extract a standardized training telemetry DataFrame.
    """
    history_obj = None

    if isinstance(ckpt_obj, dict):
        # 1. Check known top-level history dictionary keys
        for hk in [
            "history",
            "train_history",
            "training_history",
            "diagnostics",
            "metrics",
            "training_metrics",
            "train_metrics",
            "log",
            "training_log",
            "telemetry",
            "stats",
            "losses",
        ]:
            if hk in ckpt_obj and isinstance(ckpt_obj[hk], (dict, list, pd.DataFrame)):
                history_obj = ckpt_obj[hk]
                break

        # 2. Check if nested dictionaries contain metric keys
        if history_obj is None:
            for vk, vv in ckpt_obj.items():
                if isinstance(vv, dict) and any(
                    k in vv
                    for k in [
                        "total_loss",
                        "policy_loss",
                        "loss",
                        "entropy",
                        "value_loss",
                        "explained_variance",
                    ]
                ):
                    history_obj = vv
                    break

        # 3. Check if checkpoint is a direct flat dictionary of epoch lists
        if history_obj is None and any(
            k in ckpt_obj
            for k in [
                "total_loss",
                "policy_loss",
                "value_loss",
                "entropy",
                "explained_variance",
                "loss",
            ]
        ):
            history_obj = {
                k: v
                for k, v in ckpt_obj.items()
                if isinstance(v, (list, np.ndarray)) and len(v) > 0
            }

    # Convert extracted history object to DataFrame
    df_result = None
    if history_obj is not None:
        if isinstance(history_obj, pd.DataFrame):
            df_result = history_obj.copy()
        elif isinstance(history_obj, dict):
            try:
                df_result = pd.DataFrame(history_obj)
            except Exception:
                df_result = pd.DataFrame.from_dict(history_obj, orient="index")
        elif isinstance(history_obj, list):
            df_result = pd.DataFrame(history_obj)

    # 4. Companion file inspection on disk
    if (df_result is None or df_result.empty) and candidate_path is not None:
        stem = candidate_path.stem
        parent = candidate_path.parent
        companion_candidates = [
            parent / f"{stem}_history.parquet",
            parent / f"{stem}_history.csv",
            parent / f"{stem}_metrics.parquet",
            parent / f"{stem}_metrics.csv",
            parent / f"{stem}_log.csv",
            parent / "training_history.parquet",
            parent / "training_history.csv",
            parent / "training_log.csv",
            OUTPUT_DIR / "training_history.parquet",
            OUTPUT_DIR / "ppo_telemetry.parquet",
        ]
        for c_file in companion_candidates:
            if c_file.exists():
                try:
                    df_result = (
                        pd.read_parquet(c_file)
                        if c_file.suffix == ".parquet"
                        else pd.read_csv(c_file)
                    )
                    break
                except Exception:
                    continue

    return df_result


# 1. Discover all checkpoints & telemetry logs
all_disk_ckpts = sorted(
    list({p.resolve(): p for p in OUTPUT_DIR.glob("**/*.pt")}.values()),
    key=lambda p: (
        0 if "_ep_" in p.name or "model_P" in p.name else 1,
        p.stat().st_mtime,
    ),
    reverse=True,
)
all_disk_logs = sorted(
    list(OUTPUT_DIR.glob("**/*history*.parquet"))
    + list(OUTPUT_DIR.glob("**/*telemetry*.parquet"))
    + list(OUTPUT_DIR.glob("**/*train_log*.parquet"))
    + list(OUTPUT_DIR.glob("**/*history*.csv"))
    + list(OUTPUT_DIR.glob("**/*train_log*.csv"))
)

# 2. Pre-scan candidates to find verified telemetry sources
telemetry_catalog: Dict[str, Tuple[Path, Optional[pd.DataFrame]]] = {}

# Priority 1: Check checkpoints with epoch/history indicators first
for p in all_disk_ckpts:
    try:
        # Fast metadata probe
        c_dict = torch.load(p, map_location="cpu", weights_only=False)
        extracted_df = _extract_telemetry_df(c_dict, p)
        if extracted_df is not None and not extracted_df.empty:
            telemetry_catalog[p.name] = (p, extracted_df)
    except Exception:
        continue

# Priority 2: Standalone telemetry logs
for lp in all_disk_logs:
    if lp.name not in telemetry_catalog:
        try:
            df_log = pd.read_parquet(lp) if lp.suffix == ".parquet" else pd.read_csv(lp)
            if not df_log.empty:
                telemetry_catalog[lp.name] = (lp, df_log)
        except Exception:
            continue

# Determine candidate choices for dropdown
all_ckpt_names = [p.name for p in all_disk_ckpts]
catalog_names = list(telemetry_catalog.keys())
full_choices = list(dict.fromkeys(catalog_names + all_ckpt_names))

if not full_choices:
    print(
        "⚠️ No PyTorch checkpoints or telemetry logs detected across output directory."
    )
else:
    # 3. Determine Default Choice
    default_telemetry_target = None

    # Priority A: Check if a checkpoint specifically has _ep_ or model_P
    ep_matches = [
        name for name in catalog_names if "_ep_" in name or name.startswith("model_P")
    ]
    if ep_matches:
        default_telemetry_target = ep_matches[0]
    elif catalog_names:
        default_telemetry_target = catalog_names[0]
    elif full_choices:
        default_telemetry_target = full_choices[0]

    # Programmatic override handling
    if (
        DIAG_CHECKPOINT_OVERRIDE is not None
        and DIAG_CHECKPOINT_OVERRIDE in full_choices
    ):
        default_telemetry_target = DIAG_CHECKPOINT_OVERRIDE

    out_diag = widgets.Output()

    def render_diagnostics(selected_name: str):
        """Extracts data and renders the authoritative 6-panel PPO convergence telemetry dashboard."""
        target_path_loc = None
        df_diag_data = None

        if selected_name in telemetry_catalog:
            target_path_loc, df_diag_data = telemetry_catalog[selected_name]
        else:
            # Locate file on disk
            matches = list(OUTPUT_DIR.rglob(selected_name))
            if matches:
                target_path_loc = matches[0]
                try:
                    c_dict = torch.load(
                        target_path_loc, map_location=device, weights_only=False
                    )
                    df_diag_data = _extract_telemetry_df(c_dict, target_path_loc)
                except Exception as ex:
                    print(f"❌ Failed loading {selected_name}: {ex}")

        # Intelligent Fallback if selected file only has model weights
        active_display_name = selected_name
        if (
            df_diag_data is None or df_diag_data.empty
        ) and default_telemetry_target in telemetry_catalog:
            fb_path, fb_df = telemetry_catalog[default_telemetry_target]
            print(
                f"ℹ️ '{selected_name}' contains policy weights and validation metrics (no epoch arrays)."
            )
            print(
                f"⚡ Seamlessly linking active telemetry from available run: '{default_telemetry_target}'"
            )
            df_diag_data = fb_df
            active_display_name = default_telemetry_target

        if df_diag_data is None or df_diag_data.empty:
            print(f"⚠️ No training convergence telemetry found for '{selected_name}'.")
            return

        def _resolve_col(df: pd.DataFrame, candidates: List[str]) -> Optional[str]:
            lmap = {c.lower(): c for c in df.columns}
            for cand in candidates:
                if cand.lower() in lmap:
                    return lmap[cand.lower()]
            return None

        col_tot = _resolve_col(
            df_diag_data,
            [
                "total_loss",
                "loss",
                "tot_loss",
                "Total Loss",
                "loss_total",
                "train_loss",
            ],
        )
        col_pol = _resolve_col(
            df_diag_data,
            [
                "policy_loss",
                "actor_loss",
                "p_loss",
                "pi_loss",
                "Policy Loss",
                "loss_pi",
            ],
        )
        col_val = _resolve_col(
            df_diag_data,
            [
                "value_loss",
                "critic_loss",
                "v_loss",
                "val_loss",
                "Value Loss",
                "loss_v",
            ],
        )
        col_ent = _resolve_col(
            df_diag_data,
            ["entropy", "policy_entropy", "ent", "Entropy", "entropy_loss"],
        )
        col_kl = _resolve_col(
            df_diag_data,
            ["approx_kl", "kl", "kl_div", "approxkl", "Approx KL", "mean_kl"],
        )
        col_clip = _resolve_col(
            df_diag_data,
            [
                "clip_fraction",
                "clip_frac",
                "clipfrac",
                "clip_ratio",
                "Clip Fraction",
                "clip",
            ],
        )
        col_ev = _resolve_col(
            df_diag_data,
            [
                "explained_variance",
                "explained_var",
                "expl_var",
                "Explained Var",
                "ev",
                "exp_var",
            ],
        )
        col_rew = _resolve_col(
            df_diag_data,
            [
                "avg_reward",
                "mean_reward",
                "step_reward",
                "reward",
                "Average Step Reward",
                "avg_step_reward",
                "ep_reward",
            ],
        )
        col_ep = _resolve_col(
            df_diag_data,
            ["epoch", "ep", "iteration", "iter", "step", "Epoch"],
        )

        x_axis = (
            df_diag_data[col_ep].to_numpy()
            if col_ep is not None
            else np.arange(1, len(df_diag_data) + 1)
        )

        # 6-Panel Institutional PPO Telemetry Dashboard (217_diag)
        fig_diag = make_subplots(
            rows=2,
            cols=3,
            subplot_titles=(
                "Total & Policy Loss",
                "Value Loss (Critic)",
                "Entropy (Exploration)",
                "PPO Stability (Approx KL & Clip Frac)",
                "Explained Variance (Value Accuracy)",
                "Average Step Reward",
            ),
            horizontal_spacing=0.07,
            vertical_spacing=0.15,
        )

        # Panel (1, 1): Total & Policy Loss
        if col_tot is not None:
            fig_diag.add_trace(
                go.Scatter(
                    x=x_axis,
                    y=df_diag_data[col_tot],
                    mode="lines",
                    name="Total Loss",
                    line=dict(color="#636EFA", width=1.8),
                ),
                row=1,
                col=1,
            )
        if col_pol is not None:
            fig_diag.add_trace(
                go.Scatter(
                    x=x_axis,
                    y=df_diag_data[col_pol],
                    mode="lines",
                    name="Policy Loss",
                    line=dict(color="#EF553B", width=1.8),
                ),
                row=1,
                col=1,
            )

        # Panel (1, 2): Value Loss (Critic)
        if col_val is not None:
            fig_diag.add_trace(
                go.Scatter(
                    x=x_axis,
                    y=df_diag_data[col_val],
                    mode="lines",
                    name="Value Loss",
                    line=dict(color="#FFA15A", width=1.8),
                ),
                row=1,
                col=2,
            )

        # Panel (1, 3): Entropy (Exploration)
        if col_ent is not None:
            fig_diag.add_trace(
                go.Scatter(
                    x=x_axis,
                    y=df_diag_data[col_ent],
                    mode="lines",
                    name="Entropy",
                    line=dict(color="#2CA02C", width=1.8),
                ),
                row=1,
                col=3,
            )

        # Panel (2, 1): PPO Stability (Approx KL & Clip Frac)
        if col_kl is not None:
            fig_diag.add_trace(
                go.Scatter(
                    x=x_axis,
                    y=df_diag_data[col_kl],
                    mode="lines",
                    name="Approx KL",
                    line=dict(color="#008080", width=1.8),
                ),
                row=2,
                col=1,
            )
        if col_clip is not None:
            fig_diag.add_trace(
                go.Scatter(
                    x=x_axis,
                    y=df_diag_data[col_clip],
                    mode="lines",
                    name="Clip Fraction",
                    line=dict(color="#DC2626", width=2.0, dash="dot"),
                ),
                row=2,
                col=1,
            )

        # Panel (2, 2): Explained Variance (Value Accuracy)
        if col_ev is not None:
            fig_diag.add_trace(
                go.Scatter(
                    x=x_axis,
                    y=df_diag_data[col_ev],
                    mode="lines",
                    name="Explained Var",
                    line=dict(color="#8B0000", width=1.8),
                ),
                row=2,
                col=2,
            )
        fig_diag.add_hline(y=1.0, line_dash="dash", line_color="gray", row=2, col=2)
        fig_diag.add_hline(y=0.0, line_dash="dot", line_color="red", row=2, col=2)

        # Panel (2, 3): Average Step Reward
        if col_rew is not None:
            fig_diag.add_trace(
                go.Scatter(
                    x=x_axis,
                    y=df_diag_data[col_rew],
                    mode="lines",
                    name="Avg Reward",
                    line=dict(color="#7B1FA2", width=1.8),
                ),
                row=2,
                col=3,
            )
        fig_diag.update_yaxes(tickformat="~s", row=2, col=3)

        fig_diag.update_layout(
            title_text=f"🧠 Agent Training Diagnostics ({active_display_name})",
            height=650,
            width=1150,
            hovermode="x unified",
            legend=dict(
                orientation="v",
                yanchor="top",
                y=0.98,
                xanchor="left",
                x=1.02,
            ),
            margin=dict(l=50, r=160, t=80, b=50),
        )
        fig_diag.show()
        print(
            f"✅ Convergence Telemetry plotted successfully for '{active_display_name}' ({len(df_diag_data)} training steps logged)."
        )

    # Interactive Dropdown Controller
    if "diag_ckpt_selector" not in globals():
        diag_ckpt_selector = widgets.Dropdown(
            options=full_choices,
            value=default_telemetry_target,
            description="Telemetry Source:",
            layout=widgets.Layout(width="740px"),
        )

        def _on_diag_selection_change(change):
            if change.get("new"):
                with out_diag:
                    out_diag.clear_output(wait=True)
                    render_diagnostics(change["new"])

        diag_ckpt_selector.observe(_on_diag_selection_change, names="value")

    display(diag_ckpt_selector)
    display(out_diag)

    # Initial Render on Cell Execution
    with out_diag:
        render_diagnostics(diag_ckpt_selector.value)

Dropdown(description='Telemetry Source:', index=12, layout=Layout(width='740px'), options=('model_chunk3_s777_…

Output()